# 🔍 파이썬 포트 스캐너

nmap을 단순하게 만든 TCP 포트 스캐너입니다. 위에서부터 셀을 **순서대로 하나씩 실행**하면 됩니다. (`Shift + Enter`)

> ⚠️ 본인 컴퓨터(`127.0.0.1`)나 스캔이 허가된 서버(`scanme.nmap.org`)에만 사용하세요!

| 블록 | 하는 일 |
|---|---|
| 0. import | 필요한 도구 불러오기 |
| 1. `COMMON_SERVICES` | 포트 번호 → 서비스 이름 사전 |
| 2. `scan_port()` | 포트 **하나**가 열렸는지 검사 |
| 3. `service_name()` | 포트 번호로 서비스 이름 찾기 |
| 4. `parse_ports()` | `"1-1000"` 같은 글자를 포트 목록으로 변환 |
| 5. `run_scan()` | 위 함수들을 조합해서 전체 스캔 실행 |
| 6. 실행! | 실제로 스캔해보기 |

## 0. 필요한 도구 불러오기

- `socket` : 네트워크 연결을 만드는 도구 (이번 과제의 핵심)
- `time` : 시간 측정용
- `ThreadPoolExecutor` : 여러 작업을 동시에 돌리는 도구 (멀티스레딩)

In [ ]:
import socket
import time
from concurrent.futures import ThreadPoolExecutor

## 1. 서비스 이름 사전 — `COMMON_SERVICES`

"22번 포트는 보통 SSH" 처럼 **자주 쓰이는 포트 번호와 서비스 이름**을 짝지어 둔 딕셔너리입니다.
OS가 이름을 모르는 포트가 있을 때 대신 찾아볼 용도입니다.

In [ ]:
COMMON_SERVICES = {
    21: "ftp", 22: "ssh", 23: "telnet", 25: "smtp", 53: "dns",
    80: "http", 110: "pop3", 143: "imap", 443: "https", 445: "smb",
    3306: "mysql", 3389: "rdp", 5432: "postgresql", 6379: "redis", 8080: "http-alt",
}

## 2. 포트 하나 검사하기 — `scan_port()` ⭐ 핵심

1. 소켓(통신용 전화기)을 하나 만든다
2. 타임아웃을 건다 → 대답이 없으면 `timeout`초만 기다리고 포기
3. `connect_ex()`로 연결을 시도한다
   - 성공하면 `0`을 돌려줌 → **OPEN**
   - 실패하면 0이 아닌 에러 번호 → **CLOSED**

`with` 문을 쓰면 검사가 끝난 뒤 소켓이 자동으로 닫힙니다.

In [ ]:
def scan_port(host, port, timeout):
    """포트 하나에 TCP 연결을 시도. 열려 있으면 True."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        # connect_ex는 예외 대신 에러 코드를 반환한다. 0이면 연결 성공(OPEN)
        return sock.connect_ex((host, port)) == 0

In [ ]:
# 🧪 테스트: 내 컴퓨터의 135번 포트 (Windows는 보통 열려 있음)
print(scan_port("127.0.0.1", 135, 0.5))

## 3. 서비스 이름 찾기 — `service_name()`

1. 먼저 OS에게 물어본다 → `socket.getservbyport()`
2. OS가 모르면(에러) → 위에서 만든 `COMMON_SERVICES` 사전에서 찾는다
3. 거기도 없으면 → `"unknown"`

In [ ]:
def service_name(port):
    try:
        return socket.getservbyport(port, "tcp")
    except OSError:
        return COMMON_SERVICES.get(port, "unknown")

In [ ]:
# 🧪 테스트
print(service_name(22), service_name(80), service_name(3306), service_name(12345))

## 4. 포트 범위 해석하기 — `parse_ports()`

사람이 입력한 글자를 컴퓨터가 쓸 수 있는 숫자 범위로 바꿉니다.

- `"80"` → 80번 하나
- `"1-1000"` → 1번부터 1000번까지

포트 번호는 1~65535만 존재하므로 범위를 벗어나면 에러를 냅니다.

In [ ]:
def parse_ports(text):
    """'80' 또는 '1-1000' 형태를 range로 변환."""
    if "-" in text:
        start, end = map(int, text.split("-", 1))
    else:
        start = end = int(text)
    if not (1 <= start <= end <= 65535):
        raise ValueError("포트는 1-65535 범위여야 합니다")
    return range(start, end + 1)

In [ ]:
# 🧪 테스트
print(list(parse_ports("80")))
print(list(parse_ports("20-25")))

## 5. 전체 스캔 실행 — `run_scan()`

지금까지 만든 함수들을 조립하는 부분입니다. (`scanner.py`의 `main()`에 해당)

1. 도메인(예: `scanme.nmap.org`)이면 IP 주소로 바꾼다
2. 스캔 시작 시간 기록
3. `threads == 1`이면 포트를 **하나씩 차례로** 검사 (순차)
   그 외에는 `ThreadPoolExecutor`로 **여러 개를 동시에** 검사 (멀티스레드)
4. 걸린 시간 계산 후, 열린 포트만 골라서 출력

> 노트북에서는 `argparse`(명령줄 인자)를 쓸 수 없어서, 함수의 매개변수로 값을 받도록 바꿨습니다.

In [ ]:
def run_scan(host, ports_text, threads=50, timeout=1.0):
    ports = parse_ports(ports_text)

    try:
        ip = socket.gethostbyname(host)  # 도메인이면 IP로 변환
    except socket.gaierror:
        print(f"Cannot resolve host: {host}")
        return

    mode = "sequentially" if threads == 1 else f"with {threads} threads"
    print(f"Scanning {host} ({ip}) ports {ports.start}-{ports.stop - 1} {mode}...")

    start = time.time()
    if threads == 1:
        results = [scan_port(ip, p, timeout) for p in ports]
    else:
        with ThreadPoolExecutor(max_workers=threads) as pool:
            results = list(pool.map(lambda p: scan_port(ip, p, timeout), ports))
    elapsed = time.time() - start

    open_ports = [p for p, is_open in zip(ports, results) if is_open]
    for p in open_ports:
        print(f"Port {p} ({service_name(p)}): OPEN")

    print(f"Scan finished in {elapsed:.2f}s (open: {len(open_ports)}, closed: {len(ports) - len(open_ports)})")

## 6. 실행해보기 🚀

아래 값들을 바꿔가며 실행해보세요.

- `HOST` : 스캔할 대상
- `PORTS` : 포트 범위
- `THREADS` : 동시에 검사할 개수 (1이면 순차)
- `TIMEOUT` : 포트 하나당 기다리는 최대 시간(초)

> 💡 Windows에서는 닫힌 포트도 대답이 느려서 1~1000번 스캔에 약 10초 걸립니다.

In [ ]:
HOST = "127.0.0.1"
PORTS = "1-1000"
THREADS = 50
TIMEOUT = 0.5

run_scan(HOST, PORTS, THREADS, TIMEOUT)

### 순차 vs 멀티스레드 비교

1~1000번 순차 스캔은 **약 8분** 걸리므로, 여기서는 100개만 비교해봅니다.
(측정 결과: 1~1000번 순차 503.86초 vs 스레드 10.29초 → 약 49배 차이)

In [ ]:
run_scan("127.0.0.1", "1-100", threads=1, timeout=0.5)   # 순차: 약 50초
run_scan("127.0.0.1", "1-100", threads=50, timeout=0.5)  # 스레드: 약 1초